# 03 — Cost and Latency Analysis — Practical Examples

**Covers**: Cost calculator, monthly cost projections, latency benchmarker, batch API, rate limit math

In [ ]:
import os, json, time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'

MODEL_PRICING = {
    'gpt-4o-mini':       {'input': 0.150e-6, 'output': 0.600e-6},
    'gpt-4o':            {'input': 5.000e-6, 'output': 15.000e-6},
    'claude-3-5-haiku':  {'input': 0.800e-6, 'output': 4.000e-6},
    'claude-3-5-sonnet': {'input': 3.000e-6, 'output': 15.000e-6},
    'gemini-1.5-flash':  {'input': 0.075e-6, 'output': 0.300e-6},
    'gemini-1.5-pro':    {'input': 3.500e-6, 'output': 10.500e-6},
    'deepseek-v3':       {'input': 0.270e-6, 'output': 1.100e-6},
}

def calc_cost(model: str, in_tokens: int, out_tokens: int) -> float:
    p = MODEL_PRICING.get(model, MODEL_PRICING['gpt-4o-mini'])
    return in_tokens * p['input'] + out_tokens * p['output']

print('✅ Setup complete')

## 1. Per-Call Cost Calculator

In [ ]:
# Use response.usage for exact counts
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'system', 'content': 'You are a helpful assistant.'},
              {'role': 'user',   'content': 'Explain context windows in 3 sentences.'}],
    max_tokens=150
)

u = r.usage
cost = calc_cost(MODEL, u.prompt_tokens, u.completion_tokens)
print(f"Model: {MODEL}")
print(f"Input tokens:  {u.prompt_tokens:>6,}  → ${u.prompt_tokens * MODEL_PRICING[MODEL]['input']:.6f}")
print(f"Output tokens: {u.completion_tokens:>6,}  → ${u.completion_tokens * MODEL_PRICING[MODEL]['output']:.6f}")
print(f"Total cost:               ${cost:.6f}")
print(f"Per 1k calls:             ${cost*1000:.4f}")
print(f"Per 1M calls:             ${cost*1_000_000:.2f}")
print(f"\nResponse: {r.choices[0].message.content[:200]}")

## 2. Monthly Cost Projection by Scale

In [ ]:
# Agent profile: 2000 input + 400 output tokens per call
IN_T, OUT_T = 2000, 400

scales = [
    ('Prototype (100 calls/day)',    100,    30),
    ('Startup (1k calls/day)',       1_000,  30),
    ('Growth (10k calls/day)',       10_000, 30),
    ('Scale (100k calls/day)',       100_000,30),
]

print(f"\nMonthly Cost Comparison ({IN_T} input + {OUT_T} output tokens/call)")
print(f"{'Scale':<30}", end='')
for m in ['gpt-4o-mini', 'claude-3-5-haiku', 'gpt-4o', 'claude-3-5-sonnet']:
    print(f" {m[:15]:>15}", end='')
print()
print('-' * 100)
for label, calls_per_day, days in scales:
    total_calls = calls_per_day * days
    print(f"{label:<30}", end='')
    for m in ['gpt-4o-mini', 'claude-3-5-haiku', 'gpt-4o', 'claude-3-5-sonnet']:
        cost = calc_cost(m, IN_T, OUT_T) * total_calls
        print(f" ${cost:>14,.0f}", end='')
    print()

## 3. Live Latency Benchmark

In [ ]:
BENCH_PROMPT = 'List the 5 most important Python features for AI development. One sentence each.'
N_RUNS = 3

print(f"Latency benchmark ({N_RUNS} runs, same prompt)")
print('-' * 60)

for model_id in ['gpt-4o-mini']:  # Add 'gpt-4o' to compare
    latencies, token_rates = [], []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        r = client.chat.completions.create(
            model=model_id,
            messages=[{'role': 'user', 'content': BENCH_PROMPT}],
            max_tokens=150
        )
        elapsed = (time.perf_counter() - t0) * 1000
        latencies.append(elapsed)
        out_t = r.usage.completion_tokens
        token_rates.append(out_t / (elapsed / 1000))
    
    avg_ms  = sum(latencies) / N_RUNS
    avg_tps = sum(token_rates) / N_RUNS
    cost    = calc_cost(model_id, 50, r.usage.completion_tokens)
    print(f"[{model_id}]")
    print(f"  Avg latency:    {avg_ms:.0f}ms")
    print(f"  Avg throughput: {avg_tps:.0f} tokens/sec")
    print(f"  Cost per call:  ${cost:.6f}")
    print(f"  Latencies:      {[f'{l:.0f}ms' for l in latencies]}")

## 4. Model Comparison — Same Task, Different Models

In [ ]:
TASK = 'Write a Python function that checks if a number is prime. Include a docstring and type hints.'

# Compare gpt-4o vs gpt-4o-mini for coding tasks
results = {}
for model_id in ['gpt-4o-mini', 'gpt-4o']:
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=model_id,
        messages=[{'role': 'user', 'content': TASK}],
        max_tokens=200, temperature=0
    )
    elapsed = (time.perf_counter() - t0) * 1000
    results[model_id] = {
        'answer': r.choices[0].message.content,
        'tokens': r.usage.total_tokens,
        'latency_ms': elapsed,
        'cost': calc_cost(model_id, r.usage.prompt_tokens, r.usage.completion_tokens)
    }

print(f"Task: {TASK[:60]}...\n")
for m, res in results.items():
    print(f"[{m}]")
    print(f"  Tokens: {res['tokens']} | Latency: {res['latency_ms']:.0f}ms | Cost: ${res['cost']:.6f}")
    has_docstring = '"""' in res['answer'] or "'''" in res['answer']
    has_hints     = '-> bool' in res['answer'] or 'int' in res['answer']
    print(f"  Has docstring: {'✅' if has_docstring else '❌'} | Has type hints: {'✅' if has_hints else '❌'}")
    print(f"  Code preview: {res['answer'][:150]}...")
    print()

## 5. Find the Cheapest Model That Meets Quality Bar

In [ ]:
# Strategy: start cheap, escalate only if quality insufficient
QUALITY_TIERS = [
    ('gpt-4o-mini',    0.5),   # (model, quality_threshold 0-1)
    ('gpt-4o',         0.85),
]

def evaluate_quality(response: str, task: str) -> float:
    """Simple heuristic quality scoring (0-1). In production: use LLM-as-judge."""
    score = 0.0
    if 'def ' in response:     score += 0.3   # has function
    if 'return' in response:   score += 0.2   # has return
    if '"""' in response:      score += 0.2   # has docstring
    if '->' in response:       score += 0.2   # has return type hint
    if len(response) > 100:    score += 0.1   # non-trivial
    return score

MIN_ACCEPTABLE_QUALITY = 0.7

for model_id, _ in QUALITY_TIERS:
    r = client.chat.completions.create(
        model=model_id,
        messages=[{'role': 'user', 'content': TASK}],
        max_tokens=200, temperature=0
    )
    answer = r.choices[0].message.content
    quality = evaluate_quality(answer, TASK)
    cost = calc_cost(model_id, r.usage.prompt_tokens, r.usage.completion_tokens)
    ok = quality >= MIN_ACCEPTABLE_QUALITY
    print(f"[{model_id}] quality={quality:.2f} cost=${cost:.6f} {'✅ ACCEPTABLE' if ok else '⬆️ ESCALATE'}")
    if ok:
        print(f"  ✅ Using {model_id} — quality meets bar, no need to escalate")
        break

## 📌 Summary

- **`response.usage`** gives exact token counts — always log for cost tracking
- **mini/flash/haiku**: 10-50× cheaper with 80-90% quality — use by default
- **Escalation pattern**: start cheap, measure quality, escalate only when needed
- **Batch API**: submit bulk jobs, get 50% discount (non-real-time only)
- **Monthly projection**: multiply per-call cost × daily calls × 30 = budget planning
- **Rate limits**: design async pipelines to respect RPM/TPM limits